# Module 3.5: Retention & Decay

*Final stage of the **semantic memory lifecycle** (Notebooks 02–05). Episodic memory
uses TTL-based expiry instead of scoring; procedural memory uses RAG validation
instead of retention. Both are covered in Notebook 06.*

As an agent accumulates semantic memories over months, a problem emerges: **retrieval
quality degrades**. The more preferences and facts stored, the more noise competes
with signal. Eventually, asking "What hotel does Sarah prefer?" returns a mix of
genuine preferences and one-off mentions from months ago.

Simple cache heuristics (LRU, LFU) don't work — they can't distinguish a rarely-used
but critical allergy fact from frequently-retrieved but low-value noise.

> **The question**: How do we keep semantic memory useful as it grows?

## Quick Primer: How LRU and LFU Work

Before seeing why they fail for agent memory, let's understand how these
classic cache eviction strategies operate:

### LRU — Least Recently Used

LRU evicts the item that hasn't been **accessed** for the longest time.
It assumes: *"if you haven't used it recently, you probably won't need it soon."*

**How it works:**
1. Maintain an ordered list of items by last-access time
2. Every time an item is accessed, move it to the front (most-recent)
3. When at capacity, evict from the back (least-recent)

**Data structure:** Typically a doubly-linked list + hash map → O(1) access and eviction.

```
Access sequence: A, B, C, A, D, B  (capacity = 3)

After A:     [A]
After B:     [B, A]
After C:     [C, B, A]
After A:     [A, C, B]       ← A moves to front
After D:     [D, A, C]       ← B evicted (least recent)
After B:     [B, D, A]       ← C evicted (least recent)
```

### LFU — Least Frequently Used

LFU evicts the item that has been accessed the **fewest times** overall.
It assumes: *"items accessed many times are more important than those accessed rarely."*

**How it works:**
1. Maintain an access counter for each item
2. Every time an item is accessed, increment its counter
3. When at capacity, evict the item with the lowest counter (ties broken by recency)

**Data structure:** Min-heap or frequency-indexed buckets → O(1) or O(log n) eviction.

```
Access sequence: A, A, A, B, B, C  (capacity = 3)

Counters: A=3, B=2, C=1
At capacity → next insert evicts C (lowest count = 1)
```

### Why Both Fail for Agent Memory

| Strategy | Blind Spot | Agent Memory Example |
|----------|-----------|---------------------|
| **LRU** | Evicts old-but-critical items | A peanut allergy noted 6 months ago but not queried since → evicted, agent books unsafe meal |
| **LFU** | Keeps frequently-accessed noise over rare-but-vital facts | "Weather in Tokyo" retrieved 5× for chitchat outranks a 1× confirmed allergy |
| Both | No notion of **correctness** | Neither tracks whether using a memory led to good or bad outcomes |
| Both | No notion of **trust state** | A confirmed fact and an unverified guess have the same eviction priority |

The core gap: LRU/LFU only use a **single axis** (time or count) while agent
memory value depends on multiple correlated signals — trust, success, specificity,
and semantic relevance.

### Connection to Staleness Sweep (Notebook 03)

In the Staged Promotion notebook, we implemented a **`staleness_sweep`** that
demotes `trusted` memories whose `last_confirmed` timestamp exceeds a staleness
window (e.g. 180 days unconfirmed → demote to `provisional`). On the surface
this looks like LRU — it penalises items not "touched" recently.

The key differences that make it work where LRU doesn't:

| | Pure LRU | Staleness Sweep |
|---|---------|-----------------|
| **Signal** | Last *access* time | Last *confirmation* time |
| **Action** | Hard delete / evict | Soft demotion (trusted → provisional) |
| **Scope** | All items equally | Only trusted items — candidates already have low priority |
| **Recovery** | Gone forever | Re-confirmable: one new confirmation re-promotes |

The staleness sweep is a **trust-aware, reversible** cousin of LRU. It degrades
confidence in stale facts without discarding them — exactly the kind of nuance
that raw LRU/LFU cannot express. In this notebook's retention scorer, that
trust-state demotion feeds into the **Utility** dimension (weight 0.15): a
memory demoted to `provisional` by the staleness sweep scores lower than one
still `trusted`, making it a softer eviction candidate without blindly deleting it.

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys, os, json, random, math, certifi, sniffio
from datetime import datetime, timezone, timedelta

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from lifecycle_utils import MemoryItem, MemoryState, RetentionScorer
from shared.travel_agent import create_client

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Setup complete")

c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


Setup complete


## The Problem: Growth Despite Filtering

In Notebooks 02–04, we built strong guardrails for Sarah Chen's memory:

| Notebook | What It Does | Example |
|----------|-------------|---------|
| **02 – Identification** | Filters 10 conversation turns → only 5 survive as `candidate` memories | "Check flights to Chicago tomorrow" → discarded; "I always prefer window seats on long flights" → memorised |
| **03 – Staged Promotion** | Gates trust so unverified inferences can't influence decisions | Agent infers "vegetarian" → starts as `candidate`, invisible to recall until confirmed twice |
| **04 – Belief Revision** | Supersedes stale facts with SCD Type 2 audit trail | "Based in NYC" → superseded by "Moved to Paris permanently" |

These solve **quality at write-time**. But they don't solve **quantity over time**.

### The accumulation problem

Consider Sarah's graph after 12 months of real use:

- **Notebook 02** identified ~5 memories per 10-turn conversation. At 2 conversations/week,
  that's **~520 candidate memories/year** entering the pipeline.
- **Notebook 03** filters aggressively (only confirmed facts reach `trusted`), but even
  at a 30% survival rate, that's **~156 trusted preferences** after a year.
- **Notebook 04** supersedes contradicted beliefs, but non-contradicted facts just accumulate:
  "Marriott loyalty member", "vegetarian", "window seat", "budget $250/night", "home airport CDG",
  "peanut allergy", "prefers United domestic / Air France international" — each confirmed,
  each valid, each growing the graph.

When the agent calls `recall_preferences()` for a hotel booking, it doesn't need all 156
trusted facts — it needs the 3–5 most relevant. But the graph returns everything that
matches the embedding threshold, and the context window fills with preferences about
airlines, dietary needs, and seat types when the task is purely about hotels.

**Identification (NB 02) asks "should we store this?"**
**Promotion (NB 03) asks "can we trust this?"**
**Belief revision (NB 04) asks "is this still true?"**

None of them ask: **"given limited capacity, which memories are most worth keeping?"**

That's the retention problem.

### What the graph looks like after Notebooks 02–04

After running the previous notebooks, Sarah's Cosmos DB graph contains preferences like:

| Category | Content | State | Source | Age |
|----------|---------|-------|--------|-----|
| seat | "Window seat on long flights" | trusted | user_assertion | ~12 months |
| home_airport | "CDG Paris" | trusted | user_assertion (revised from JFK) | ~5 months |
| hotel_chain | "Marriott loyalty member" | trusted | user_assertion | ~12 months |
| diet | "Vegetarian — doesn't eat meat" | trusted | llm_inference (confirmed 2×) | ~10 months |
| budget | "Hotel budget max $250/night" | trusted | user_assertion | ~12 months |
| allergy | "Severe peanut allergy — always flag for meal selection" | trusted | user_assertion | ~12 months |
| airline | "Prefers United domestic, Air France international" | provisional | llm_inference (confirmed 1×) | ~8 months |

All of these passed identification (NB 02), earned trust (NB 03), and haven't been
contradicted (NB 04). They're all legitimate. But when the agent handles a hotel
booking query, retrieving the allergy, airline, and seat preferences alongside hotel
preferences adds noise to the context — not because they're wrong, but because
they're irrelevant to *this* task.

Now multiply by 12 months. The `staleness_sweep` from Notebook 03 demotes memories
unconfirmed for 180 days, but frequently-confirmed preferences (like "vegetarian")
never go stale — they just accumulate. Belief revision retires contradicted facts,
but what about 50 non-contradicted facts competing for context window space?

We need a mechanism that asks: **"which of these valid, trusted memories deserve
the limited retrieval slots right now?"**

## What the Previous Notebooks Don't Solve

| Lifecycle Stage | What It Handles | What It Doesn't Handle |
|-----------------|----------------|----------------------|
| **Identification (NB 02)** | Filters noise at ingest (10 turns → 5 memories) | Doesn't limit total accumulation over months |
| **Promotion (NB 03)** | Prevents unverified facts from influencing agent | Trusted facts still grow unbounded |
| **Staleness Sweep (NB 03)** | Demotes unconfirmed memories after 180 days | Frequently-confirmed memories never demote |
| **Belief Revision (NB 04)** | Retires contradicted beliefs | Non-contradicted facts accumulate forever |

The core issue: **all four mechanisms are write-path or trust-path controls.
None of them manage read-path capacity** — i.e., how many memories compete
for the agent's limited context window at retrieval time.

> *"Selective Memory Retention for Long-Horizon LLM Agents"* (arXiv:2606.29178) —
> Under 75% noise injection, unbounded memory degrades Precision@5 from
> 20.2% to 12.4%, while bounded retention maintains 16.9% → 16.6%.

## The Solution: Multi-Dimensional Scoring with Bounded Capacity

Instead of treating all memories equally, we **score** them on multiple dimensions
and evict the lowest-scored when at capacity. This is NOT a simple cache (LRU/LFU)
— it considers semantic value, success correlation, and trust state.

| Dimension | Weight | Meaning |
|-----------|--------|----------|
| Recency | 0.20 | Newer memories score higher (exponential decay) |
| Access frequency | 0.15 | Frequently retrieved memories score higher |
| Success correlation | 0.25 | Memories that led to good outcomes score highest |
| Redundancy penalty | 0.15 | Similar-to-others memories score lower |
| Specificity | 0.10 | Detailed, specific memories score higher |
| Utility (state-based) | 0.15 | Trusted > Provisional > Candidate |

In [3]:
scorer = RetentionScorer(
    capacity=20,
    half_life_days=30.0,
)

print(f"RetentionScorer configuration:")
print(f"  Capacity: {scorer.capacity}")
print(f"  Half-life: {scorer.half_life_days} days")
print(f"  Weights:")
for dim, weight in scorer.weights.items():
    print(f"    {dim:<20} = {weight:.2f}")

RetentionScorer configuration:
  Capacity: 20
  Half-life: 30.0 days
  Weights:
    recency              = 0.20
    access_frequency     = 0.15
    success              = 0.25
    redundancy_penalty   = 0.15
    specificity          = 0.10
    utility              = 0.15


In [4]:
# Create sample memories with varying characteristics
now = datetime.now(timezone.utc)

sample_memories = [
    MemoryItem(user_id="E001", content="Prefers Marriott hotels in NYC",
              category="preference", state=MemoryState.TRUSTED,
              first_seen=now - timedelta(days=10), access_count=8,
              last_accessed=now - timedelta(days=1),
              success_correlation=0.9, confirmation_count=4),
    MemoryItem(user_id="E001", content="Usually takes morning flights",
              category="preference", state=MemoryState.PROVISIONAL,
              first_seen=now - timedelta(days=45), access_count=3,
              last_accessed=now - timedelta(days=15),
              success_correlation=0.4, confirmation_count=1),
    MemoryItem(user_id="E001", content="Mentioned liking jazz music once",
              category="preference", state=MemoryState.CANDIDATE,
              first_seen=now - timedelta(days=120), access_count=0,
              last_accessed=None,
              success_correlation=0.0, confirmation_count=0),
]

In [5]:
# Score each memory to see how the dimensions combine
print("=== Retention Scores ===")
print()
for m in sample_memories:
    score = scorer.score(m)
    age_days = (now - m.first_seen).days
    print(f"  Score: {score:.3f} | {m.content[:50]}")
    print(f"          Age: {age_days}d | Accessed: {m.access_count}x | "
          f"State: {m.state.value} | Success: {m.success_correlation}")
    print()

=== Retention Scores ===

  Score: 0.666 | Prefers Marriott hotels in NYC
          Age: 10d | Accessed: 8x | State: trusted | Success: 0.9

  Score: 0.307 | Usually takes morning flights
          Age: 45d | Accessed: 3x | State: provisional | Success: 0.4

  Score: 0.059 | Mentioned liking jazz music once
          Age: 120d | Accessed: 0x | State: candidate | Success: 0.0



## Cosmos DB Integration: Retention on the Real Graph

The scoring above works on in-memory objects. But in production, preferences
live in Cosmos DB (same graph as Modules 3.3–3.4). The `CosmosRetentionStore`
reads preferences from Cosmos DB, scores them, writes scores back as node
properties, and evicts (→ DEPRECATED) the lowest-scored.

This composes with promotion and belief revision — retention never touches
`valid_from`/`valid_to` or `confirmation_count`. It only manages capacity.

In [ ]:
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider)
from openai import AsyncAzureOpenAI
from azure.cosmos.aio import CosmosClient
from shared.semantic_store import SemanticMemoryStore, create_container

embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(), "https://cognitiveservices.azure.com/.default")
embed_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
embed_client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token, api_version="2024-02-01")


async def embed(text: str) -> list[float]:print("Embedding provider ready")

    r = await embed_client.embeddings.create(input=[text], model=embed_deployment)
    return r.data[0].embedding

LLM provider ready


In [ ]:
embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(), "https://cognitiveservices.azure.com/.default")
embed_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
embed_client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token, api_version="2024-02-01")
print("(embedding provider already initialized above)")

Embedding provider ready


In [ ]:
from lifecycle_utils import CosmosRetentionStore

cosmos = CosmosClient(os.environ["COSMOS_ENDPOINT"], credential=AsyncCliCredential())
container = await create_container(cosmos)

retention = CosmosRetentionStore(
    SemanticMemoryStore(container, user_id="E001", embed_fn=embed,
                        scope_tag="module_03_retention_demo"),
    capacity=12,
)
removed_stale_demo = await retention.cleanup_scope()
print(f"CosmosRetentionStore ready (capacity={retention.capacity})")
print(f"Connected to Cosmos DB: semantic-memory")
print(f"Removed {removed_stale_demo} leftover demo preferences from an earlier run")

CosmosRetentionStore ready (capacity=12)
Connected to Cosmos DB: Cosmos DB+s://892e1c49.databases.Cosmos DB.io
Removed 0 leftover demo preferences from an earlier run


## Populating Cosmos DB: Signal + Noise

A long-running assistant would accumulate these records gradually across months of conversations. To reproduce that growth without replaying hundreds of turns, we add 20 **synthetic demo preferences**: high-value signal (trusted, frequently accessed, successful) and low-value noise (candidate, never accessed, zero success).

Every synthetic preference is tagged with an isolated demo scope. The scorer and eviction job operate only within that scope, and a cleanup cell removes the fixtures afterward. Memories created by earlier notebooks remain untouched.

In [9]:
# High-value signal preferences (should survive eviction)
SIGNAL = [
    ("hotel_chain", "Prefers Marriott hotels worldwide", "trusted", 8, 0.9, 10),
    ("diet", "Strict vegetarian — no meat in meals", "trusted", 6, 0.85, 15),
    ("seat", "Window seat on flights over 4 hours", "trusted", 5, 0.8, 20),
    ("home_airport", "Home airport is CDG Paris", "trusted", 10, 0.95, 5),
    ("budget", "Hotel budget max $350/night", "provisional", 4, 0.7, 30),
    ("airline", "Prefers United for domestic, Air France for international",
     "provisional", 3, 0.6, 25),
    ("allergy", "Severe peanut allergy — requires airline notification 48h before flight",
     "trusted", 5, 0.9, 180),
]
print(f"Signal preferences to write: {len(SIGNAL)}")

Signal preferences to write: 7


In [10]:
# Low-value noise (should get evicted)
NOISE = [
    ("weather", "Asked about weather in Tokyo", "candidate", 0, 0.0, 90),
    ("misc", "Colleague mentioned Hilton once", "candidate", 0, 0.0, 120),
    ("misc", "Saw an ad for Spirit Airlines", "candidate", 0, 0.05, 100),
    ("misc", "Discussed rental car prices briefly", "candidate", 0, 0.0, 85),
    ("misc", "Random observation about JFK airport", "candidate", 0, 0.1, 110),
    ("misc", "Asked about train passes in Europe", "candidate", 1, 0.0, 95),
    ("weather", "Weather was cold in London", "candidate", 0, 0.0, 75),
    ("misc", "Mentioned duty free shopping", "candidate", 0, 0.0, 130),
    ("misc", "Colleague recommended travel insurance", "candidate", 0, 0.05, 60),
    ("misc", "Talked about airport lounges once", "candidate", 1, 0.1, 80),
    ("misc", "Asked about currency exchange rates", "candidate", 0, 0.0, 70),
    ("misc", "Discussed commute times to airport", "candidate", 0, 0.0, 140),
    ("misc", "Asked about pet travel policies", "candidate", 0, 0.0, 150),
]
print(f"Noise preferences to write: {len(NOISE)}")
print(f"Total: {len(SIGNAL) + len(NOISE)} (capacity: {retention.capacity})")

Noise preferences to write: 13
Total: 20 (capacity: 12)


In [11]:
# Write all preferences to Cosmos DB
for cat, pref, state, access, success, age in SIGNAL + NOISE:
    await retention.add_preference(
        category=cat, preference=pref, state=state,
        access_count=access, success_correlation=success, age_days=age)

total = await retention.active_count()
print(f"\nCosmos DB now has {total} active preferences (capacity: {retention.capacity})")
print(f"Over capacity by: {total - retention.capacity}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replac


Cosmos DB now has 20 active preferences (capacity: 12)
Over capacity by: 8


## Before Eviction: What Does Search Return?

With 20 preferences stored, a semantic search for "hotel" returns both
signal and noise. The agent would see irrelevant results mixed in.

In [ ]:
# Search before eviction
pre_results = await retention._s.search(
    "hotel and accommodation preferences", top_k=8)

print("=== Pre-Eviction Search: 'hotel preferences' ===")
print()
for p in pre_results:
    print(f"  {p['preference'][:60]}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\nRETURN node AS p, score\nORDER BY score DESC\n"


=== Pre-Eviction Search: 'hotel preferences' ===

  Marriott
  Hotel budget max $350/night
  Asked about pet travel policies
  Prefers United for domestic, Air France for international
  Discussed commute times to airport
  Colleague recommended travel insurance
  Colleague mentioned Hilton once
  Asked about train passes in Europe


## Running Retention Scoring

`score_all()` evaluates every active preference in this demo scope and writes the resulting score back to Cosmos DB.

### Keep retention off the agent's hot path

Retention scoring and eviction should run as a **separate background maintenance process**, such as an hourly or nightly scheduled job—not inside the agent's conversational turn. Promotion and belief revision protect the write and recall paths in real time; retention periodically manages capacity after memories accumulate.

This separation avoids adding a full-memory scan to user-facing latency and keeps deterministic maintenance independent from the LLM. The notebook invokes the job directly only so its effects are visible step by step.

In [13]:
# Score all preferences — writes retention_score back to Cosmos DB nodes
scored = await retention.score_all()

print(f"=== Retention Scores (ascending = evict first) ===")
print(f"{'Score':<8} {'State':<12} {'Preference'}")
print("\u2500" * 70)
for item in scored:
    print(f"{item['score']:<8.3f} {item['state']:<12} {item['preference'][:50]}")

=== Retention Scores (ascending = evict first) ===
Score    State        Preference
──────────────────────────────────────────────────────────────────────
0.052    candidate    Asked about pet travel policies
0.054    candidate    Mentioned duty free shopping
0.055    candidate    Discussed commute times to airport
0.058    candidate    Colleague mentioned Hilton once
0.069    candidate    Asked about weather in Tokyo
0.076    candidate    Discussed rental car prices briefly
0.077    candidate    Saw an ad for Spirit Airlines
0.077    candidate    Asked about train passes in Europe
0.078    candidate    Weather was cold in London
0.087    candidate    Asked about currency exchange rates
0.089    candidate    Random observation about JFK airport
0.112    candidate    Colleague recommended travel insurance
0.112    candidate    Talked about airport lounges once
0.404    provisional  Prefers United for domestic, Air France for intern
0.405    trusted      Severe peanut allergy — requires 

## Eviction: Deprecate the Lowest-Scored

We're over capacity (20 preferences, capacity 12). Evict the bottom 8.
Eviction = set `state` to `deprecated` — the preference stays in Cosmos DB
for audit but is excluded from future searches.

In [14]:
overflow = (await retention.active_count()) - retention.capacity
evicted = await retention.evict(count=overflow)

print(f"=== Evicted {len(evicted)} Preferences ===")
print()
for item in evicted:
    print(f"  Score: {item['score']:.3f} | {item['preference'][:55]}")

remaining = await retention.active_count()
print(f"\nActive preferences remaining: {remaining}/{retention.capacity}")

=== Evicted 8 Preferences ===

  Score: 0.052 | Asked about pet travel policies
  Score: 0.054 | Mentioned duty free shopping
  Score: 0.055 | Discussed commute times to airport
  Score: 0.058 | Colleague mentioned Hilton once
  Score: 0.069 | Asked about weather in Tokyo
  Score: 0.076 | Discussed rental car prices briefly
  Score: 0.077 | Saw an ad for Spirit Airlines
  Score: 0.077 | Asked about train passes in Europe

Active preferences remaining: 12/12


## After Eviction: Cleaner Search Results

Now the same search returns only relevant, high-value preferences.
Noise has been deprecated and won't appear in the agent's recall path.

In [15]:
# What survived? Show the full snapshot
snap = await retention.snapshot()

print("=== Post-Eviction: Surviving Preferences ===")
print()
for p in snap:
    if p["state"] != "deprecated":
        score = p.get('score') or 0
        print(f"  [{p['state']:<11}] Score: {score:.3f} | {p['preference'][:50]}")

print(f"\n\u2192 All signal preferences retained; noise evicted")

=== Post-Eviction: Surviving Preferences ===

  [trusted    ] Score: 0.684 | CDG
  [trusted    ] Score: 0.620 | Marriott
  [trusted    ] Score: 0.574 | Strict vegetarian — no meat in meals
  [trusted    ] Score: 0.529 | Window seat on flights over 4 hours
  [provisional] Score: 0.412 | Hotel budget max $350/night
  [trusted    ] Score: 0.405 | Severe peanut allergy — requires airline notificat
  [provisional] Score: 0.404 | Prefers United for domestic, Air France for intern
  [candidate  ] Score: 0.112 | Talked about airport lounges once
  [candidate  ] Score: 0.112 | Colleague recommended travel insurance
  [candidate  ] Score: 0.089 | Random observation about JFK airport
  [candidate  ] Score: 0.087 | Asked about currency exchange rates
  [candidate  ] Score: 0.078 | Weather was cold in London

→ All signal preferences retained; noise evicted


## Key Insight: Why the Allergy Survived

The peanut allergy fact is **180 days old** — older than most noise. An LRU
cache would have evicted it. But the retention scorer keeps it because:

- **Success correlation**: 0.9 (using it led to good outcomes)
- **State**: TRUSTED (highest utility tier)
- **Specificity**: long, detailed content
- **Access count**: 5 (regularly retrieved)

These signals outweigh the recency penalty. This is why multi-dimensional
scoring beats simple cache heuristics for agent memory.

## Key Takeaways

1. **Bounded > Unbounded** — capacity limits prevent noise accumulation
2. **Multi-dimensional scoring** — not just recency (LRU) or frequency (LFU)
3. **Success correlation is the strongest signal** — weight 0.25 (highest)
4. **Critical facts survive age** — allergy info kept despite being 180 days old
5. **Eviction = DEPRECATED, not deleted** — audit trail preserved in Cosmos DB
6. **Same graph, new properties** — retention scores live alongside promotion and belief data
7. **Recency uses exponential decay** — 30-day half-life means a memory is worth 50% at 30 days, 25% at 60, negligible by 120
8. **Retention runs outside the agent** — schedule scoring and eviction as background maintenance rather than adding them to each conversational turn

## Semantic Memory Lifecycle Complete

Notebooks 01–05 form the complete **semantic memory lifecycle**:

| Notebook | Capability |
|----------|------------|
| 01 | Distinguished memory from RAG and context |
| 02 | Identified what qualifies as semantic memory |
| 03 | Gated trust with staged promotion |
| 04 | Revised beliefs with bi-temporal tracking |
| 05 | Bounded retention with score-based eviction |

Together: **identify → promote → revise → retain/evict**.

In Module 10 (Unified Agent), promotion and belief revision compose into the agent-facing memory interface, while retention remains a separately scheduled maintenance operation over the same graph.

## Next: Episodic & Procedural Lifecycles (Notebook 06)

Semantic memory (preferences, facts) has the lifecycle above. But episodic memory (past events) and procedural memory (learned procedures) need their own lifecycle strategies — TTL-based expiry, cross-session graduation, and RAG-validated staleness.

## Cleanup: Remove Only the Synthetic Preferences

The retention experiment is complete. Remove the scoped fixtures so they cannot affect Module 3's earlier lifecycle memories or the Module 4 provenance exercises.

In [ ]:
removed = await retention.cleanup_scope()
preserved = await retention._s.count(current_only=False)
print(f"Removed {removed} synthetic retention-demo preferences")
print(f"Preserved {preserved} preferences from earlier notebooks")

Removed 20 synthetic retention-demo preferences
Preserved 3 preferences from earlier notebooks
